# Quick Start: PC-SAFT Parameter Prediction

This notebook demonstrates the basic usage of `pcsaft-predict` for predicting PC-SAFT equation-of-state parameters from SMILES strings.

**What you'll learn**:
1. Install and import the package
2. Predict parameters for diverse molecules
3. Visualize uncertainty estimates
4. Check applicability domain
5. Compare models
6. Batch prediction from CSV

**Runtime**: < 2 minutes (with models pre-downloaded)

## 1. Installation and Import

Install the package if not already installed:

In [ ]:
# Uncomment to install
# !pip install pcsaft-predict

import matplotlib.pyplot as plt
import pandas as pd

import pcsaft_predict

print(f"pcsaft-predict version: {pcsaft_predict.__version__}")
print(f"Available models: {pcsaft_predict.list_models()}")

## 2. Basic Prediction: 5 Diverse Molecules

Let's predict PC-SAFT parameters for structurally diverse molecules:

In [ ]:
# Define test molecules
molecules = {
    "Ethanol": "CCO",
    "Benzene": "c1ccccc1",
    "Isobutane": "CC(C)C",
    "1-Fluorocyclopentene": "FC1=CCCC1",
    "Cyclopentane": "C1CCCC1"
}

smiles_list = list(molecules.values())
names = list(molecules.keys())

# Predict
df = pcsaft_predict.predict(smiles_list)
df.insert(0, "name", names)

# Display results
print("\nPC-SAFT Parameter Predictions:")
print("=" * 80)
print(df[["name", "m", "sigma", "epsilon_k", "in_domain", "tanimoto_nn"]].to_string(index=False))

**Interpretation**:
- `m`: Number of segments (higher = longer/heavier molecule)
- `sigma`: Segment diameter in Angstroms (molecular size)
- `epsilon_k`: Dispersion energy in Kelvin (intermolecular attraction strength)
- `in_domain`: TRUE if Tanimoto similarity to training set ≥ 0.4
- `tanimoto_nn`: Similarity to nearest training molecule (0-1 scale)

## 3. Uncertainty Quantification

Get predictions with uncertainty estimates from Random Forest tree variance:

In [ ]:
# Predict with uncertainty
df_unc = pcsaft_predict.predict_with_uncertainty(smiles_list)
df_unc.insert(0, "name", names)

# Display with uncertainty columns
print("\nPredictions with Uncertainty:")
print("=" * 100)
display_cols = [
    "name", "m", "m_std", "sigma", "sigma_std",
    "epsilon_k", "epsilon_k_std", "in_domain",
]
print(df_unc[display_cols].to_string(index=False))

# Flag high-uncertainty predictions
high_unc = df_unc[df_unc["epsilon_k_std"] > 15.0]
if len(high_unc) > 0:
    n = len(high_unc)
    print(f"\n{n} molecule(s) with high uncertainty (eps/k std > 15 K):")
    print(high_unc[["name", "epsilon_k", "epsilon_k_std"]].to_string(index=False))
else:
    print("\nAll predictions have acceptable uncertainty (eps/k std <= 15 K)")

## 4. Visualize Uncertainty

Plot predictions with error bars:

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

params = [
    ("m", "Number of Segments", "m"),
    ("sigma", "Segment Diameter", "sigma (A)"),
    ("epsilon_k", "Dispersion Energy", "eps/k (K)")
]

for ax, (param, title, ylabel) in zip(axes, params):
    x = range(len(df_unc))
    y = df_unc[param]
    yerr = df_unc[f"{param}_std"]

    # Color by in_domain
    colors = [
        "green" if d else "orange" for d in df_unc["in_domain"]
    ]

    ax.errorbar(
        x, y, yerr=yerr, fmt="o", markersize=8,
        capsize=5, color="black", elinewidth=2,
    )
    ax.scatter(
        x, y, c=colors, s=100, zorder=5,
        edgecolors="black", linewidth=1.5,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(df_unc["name"], rotation=45, ha="right")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(alpha=0.3, linestyle="--")

# Legend
legend_elements = [
    Patch(facecolor="green", label="In domain (Tanimoto >= 0.4)"),
    Patch(facecolor="orange", label="Out-of-domain (Tanimoto < 0.4)"),
]
fig.legend(
    handles=legend_elements, loc="upper center",
    ncol=2, bbox_to_anchor=(0.5, 0.0), fontsize=11,
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 5. Applicability Domain Analysis

Check which molecules are within the training distribution:

In [ ]:
print("\nApplicability Domain Summary:")
print("=" * 80)

for idx, row in df_unc.iterrows():
    name = row["name"]
    tanimoto = row["tanimoto_nn"]
    in_domain = row["in_domain"]
    
    status = "✓ IN DOMAIN" if in_domain else "⚠️  OUT-OF-DOMAIN"
    confidence = "High" if tanimoto >= 0.6 else "Medium" if tanimoto >= 0.4 else "Low"
    
    print(f"{name:25s} | Tanimoto: {tanimoto:.3f} | {status:20s} | Confidence: {confidence}")

print(f"\nTotal in-domain: {df_unc['in_domain'].sum()}/{len(df_unc)}")
print("\nGuidance:")
print("  • Tanimoto ≥ 0.6: High confidence (structurally similar to training data)")
print("  • Tanimoto 0.4-0.6: Medium confidence (acceptable similarity)")
print("  • Tanimoto < 0.4: Low confidence (extrapolation, experimental validation recommended)")

## 6. Model Comparison (Future)

When multiple models are available, compare their predictions:

In [ ]:
available_models = pcsaft_predict.list_models()
print(f"Available models: {available_models}")

# Currently only RF is packaged
# When GNN and ensemble are added, this will compare all three:

# comparison_results = []
# for model_name in available_models:
#     df_model = pcsaft_predict.predict(smiles_list, model=model_name)
#     df_model["model"] = model_name
#     comparison_results.append(df_model)
# 
# df_comparison = pd.concat(comparison_results)
# print(df_comparison[["model", "smiles", "m", "sigma", "epsilon_k"]])

print("\nNote: Only RF model is currently packaged. GNN and ensemble coming in v1.1.0.")

## 7. Batch Prediction from CSV

Process a CSV file with SMILES strings:

In [ ]:
# Create example input CSV
import io

csv_data = """smiles,name
CCO,Ethanol
c1ccccc1,Benzene
CC(C)C,Isobutane
C1CCCC1,Cyclopentane
FC(F)(F)C(F)(F)F,Hexafluoroethane
CC(=O)C,Acetone
CCCCCC,n-Hexane
"""

# Read CSV
df_input = pd.read_csv(io.StringIO(csv_data))

# Predict for all molecules
smiles_batch = df_input["smiles"].tolist()
df_predictions = pcsaft_predict.predict_with_uncertainty(smiles_batch)

# Merge with input names
df_result = pd.concat([df_input, df_predictions.drop(columns=["smiles"])], axis=1)

print("\nBatch Prediction Results:")
print("=" * 100)
display_cols = ["name", "smiles", "m", "m_std", "sigma", "epsilon_k", "in_domain"]
print(df_result[display_cols].to_string(index=False))

# Save to CSV
# df_result.to_csv("pcsaft_predictions.csv", index=False)
# print("\nResults saved to pcsaft_predictions.csv")

## 8. Summary and Next Steps

**What we covered**:
- ✓ Basic prediction with `predict()`
- ✓ Uncertainty quantification with `predict_with_uncertainty()`
- ✓ Applicability domain checking via Tanimoto similarity
- ✓ Visualization of predictions with error bars
- ✓ Batch processing from CSV files

**Next steps**:
1. **Screening workflow**: See `screening_workflow.ipynb` for a full refrigerant screening example
2. **API reference**: Read `docs/api/index.md` for detailed function documentation
3. **Contributing**: Add your own experimental data or models (see `CONTRIBUTING.md`)

**Key takeaways**:
- Always check `in_domain` flag and `tanimoto_nn` before trusting predictions
- High `epsilon_k_std` (> 20 K) indicates high uncertainty → experimental validation recommended
- Models are trained on ~10% fluorinated compounds → fluorinated molecules are often out-of-domain
- Use predictions for **screening and prioritization**, not direct engineering design